# Data Cleaning and Text Preprocessing

This notebook performs the complete preparation pipeline in one place:

1. Load the merged review dataset.
2. Apply general data cleaning with `clean.py`.
3. Create classical and transformer text features with `text_preprocessing.py`.
4. Validate the final dataset.
5. Save one processed CSV file.

The final output is:

```text
data/processed/processed_reviews.csv
```

## Downstream usage

For classical machine-learning models:

```python
X = df_processed["classical_text"]
y = df_processed["rating"]
```

For transformer models:

```python
X = df_processed["transformer_text"]
y = df_processed["rating"]
```

Transformer tokenization, padding, truncation, and attention-mask creation should happen later in the transformer modeling notebook.

In [1]:
# locate the project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

from src.data.text_preprocessing import (
    create_model_text_features,
    ensure_nltk_resources,
    validate_model_text_features,
)
from src.data.clean import clean_dataset, validate_clean_dataset
from src.data.load import load_csv
from src.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR


## Locate the project root

This allows the notebook to work when it is executed from the `notebooks` directory.

In [3]:
# load the interim merged dataset
merged_path = INTERIM_DATA_DIR / "merged_reviews.csv"

merged_df = load_csv(merged_path)
merged_df.head()

print(f"Raw dataset shape: {merged_df.shape}")
display(merged_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/karima/Ironhack-challenges/data/interim/merged_reviews.csv'

## Apply general data cleaning

The original DataFrame is preserved. Cleaning is applied to a copy.

In [ ]:
df_clean = clean_dataset(merged_df.copy())

validate_clean_dataset(df_clean)

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Cleaning validation passed.")

display(df_clean.head())

Cleaned dataset shape: (67853, 10)
Cleaning validation passed.


,product_name,asins,brand,categories,review_date,rating,review_text,review_title,source_dataset,primary_categories
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets",2017-01-13 00:00:00+00:00,5,This product so far has not disappointed. My children love to use it and I like the ability to monitor control what ...,Kindle,1429_1,NaN
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets",2017-01-13 00:00:00+00:00,5,great for beginner or experienced person. Bought as a gift and she loves it,very fast,1429_1,NaN
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets",2017-01-13 00:00:00+00:00,5,"Inexpensive tablet for him to use and learn on, step up from the NABI. He was thrilled with it, learn how to Skype o...",Beginner tablet for our 9 year old son.,1429_1,NaN
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets",2017-01-13 00:00:00+00:00,4,I've had my Fire HD 8 two weeks now and I love it. This tablet is a great value.We are Prime Members and that is whe...,Good!!!,1429_1,NaN
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets",2017-01-12 00:00:00+00:00,5,"I bought this for my grand daughter when she comes over to visit. I set it up with her as the user, entered her age ...",Fantastic Tablet for kids,1429_1,NaN


## Create model-specific text features

The preprocessing pipeline creates:

- `normalized_combined_text`
- `classical_text`
- `transformer_text`

Normalized duplicate removal happens before the classical and transformer branches, which keeps both columns aligned.

In [ ]:
ensure_nltk_resources()

df_processed = create_model_text_features(
    df_clean.copy(),
    remove_duplicates=True,
    include_rating_in_duplicate_key=True,
)

validate_model_text_features(df_processed)

print(f"Processed dataset shape: {df_processed.shape}")
print("Text preprocessing validation passed.")

Processed dataset shape: (47279, 13)
Text preprocessing validation passed.


## Preview the final text columns

In [ ]:
preview_columns = [
   "asins",
    "product_name",
    "brand",
    "primary_categories",
    "review_title",
    "review_text",
    "normalized_combined_text",
    "classical_text",
    "transformer_text",
    "rating",
    "sentiment_label"
]

display(df_processed[preview_columns].head())

,asins,product_name,brand,primary_categories,review_title,review_text,normalized_combined_text,classical_text,transformer_text,rating
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",Amazon,NaN,Kindle,This product so far has not disappointed. My children love to use it and I like the ability to monitor control what ...,Kindle This product so far has not disappointed. My children love to use it and I like the ability to monitor contro...,kindle product far not disappointed child love use like ability monitor control content see ease,Kindle This product so far has not disappointed. My children love to use it and I like the ability to monitor contro...,5
1,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",Amazon,NaN,very fast,great for beginner or experienced person. Bought as a gift and she loves it,very fast great for beginner or experienced person. Bought as a gift and she loves it,fast great beginner experienced person bought gift love,very fast great for beginner or experienced person. Bought as a gift and she loves it,5
2,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",Amazon,NaN,Beginner tablet for our 9 year old son.,"Inexpensive tablet for him to use and learn on, step up from the NABI. He was thrilled with it, learn how to Skype o...","Beginner tablet for our 9 year old son. Inexpensive tablet for him to use and learn on, step up from the NABI. He wa...",beginner tablet year old son inexpensive tablet use learn step nabi thrilled learn skype already,"Beginner tablet for our 9 year old son. Inexpensive tablet for him to use and learn on, step up from the NABI. He wa...",5
3,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",Amazon,NaN,Good!!!,I've had my Fire HD 8 two weeks now and I love it. This tablet is a great value.We are Prime Members and that is whe...,Good!!! I've had my Fire HD 8 two weeks now and I love it. This tablet is a great value.We are Prime Members and tha...,good fire hd two week love tablet great value prime member tablet shine love able easily access prime content well m...,Good!!! I've had my Fire HD 8 two weeks now and I love it. This tablet is a great value.We are Prime Members and tha...,4
4,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",Amazon,NaN,Fantastic Tablet for kids,"I bought this for my grand daughter when she comes over to visit. I set it up with her as the user, entered her age ...",Fantastic Tablet for kids I bought this for my grand daughter when she comes over to visit. I set it up with her as ...,fantastic tablet kid bought grand daughter come visit set user entered age name amazon make sure access site content...,Fantastic Tablet for kids I bought this for my grand daughter when she comes over to visit. I set it up with her as ...,5


## Final quality checks

In [ ]:
quality_summary = pd.Series(
    {
        "raw_rows": len(merged_df),
        "cleaned_rows": len(df_clean),
        "processed_rows": len(df_processed),
        "removed_during_cleaning": len(merged_df) - len(df_clean),
        "removed_during_normalized_deduplication": len(df_clean) - len(df_processed),
        "missing_review_text": int(df_processed["review_text"].isna().sum()),
        "invalid_ratings": int(
            (~df_processed["rating"].between(1, 5, inclusive="both")).sum()
        ),
        "empty_transformer_text": int(
            df_processed["transformer_text"]
            .astype("string")
            .str.len()
            .eq(0)
            .sum()
        ),
        "empty_classical_text": int(
            df_processed["classical_text"]
            .astype("string")
            .str.len()
            .eq(0)
            .sum()
        ),
    },
    name="quality_summary",
)

quality_summary

raw_rows                                   67992
cleaned_rows                               67853
processed_rows                             47279
removed_during_cleaning                      139
removed_during_normalized_deduplication    20574
missing_review_text                            0
invalid_ratings                                0
empty_transformer_text                         0
empty_classical_text                           3
Name: quality_summary, dtype: int64

## Save one processed CSV file

In [ ]:
processed_path = PROCESSED_DATA_DIR / "cleaned_reviews.csv"

df_processed.to_csv(processed_path, index=False)

print(f"Saved processed dataset to: {processed_path}")

Saved processed dataset to: /Users/karima/Ironhack-challenges/voxforge-ai-review-analytics/data/processed/cleaned_reviews.csv


## Final verification

In [ ]:
assert processed_path.exists()
assert "classical_text" in df_processed.columns
assert "transformer_text" in df_processed.columns
assert len(df_processed["classical_text"]) == len(df_processed["transformer_text"])

print("Final dataset is ready for modeling.")

Final dataset is ready for modeling.
